# Examples of linear fits for RM and binned RMs versus longitude
## Figures 4 and 13 in the paper
### A. Ordog, Sept 3, 2024

In [ ]:
from astropy.io import fits
from astropy.coordinates import SkyCoord
from astropy.coordinates import ICRS, Galactic, FK4, FK5  # Low-level frames
import astropy.units as u
import numpy as np
import matplotlib.pyplot as plt
from reproject.mosaicking import find_optimal_celestial_wcs
from reproject import reproject_interp
from reproject.mosaicking import reproject_and_coadd
from astropy.wcs import WCS
from astropy.utils.data import get_pkg_data_filename
from astropy.convolution import Gaussian2DKernel
from astropy.convolution import convolve
import gc
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy.stats import pearsonr

## Choose data directory, and thresholds for PI and error in RM for Figure 13

In [ ]:
dir_in  = '/srv/data/cgps-gmims/conv_regrid/'
#dir_in  = '/srv/data/cgps-gmims_2023/'

P_thr   = 0.1 # K
dRM_thr = 150 # rad/m^2

## Functions:

In [ ]:
def read_st_catalogs(directory, file):
    
    catalog = fits.open(directory+file)
    
    st_cat = {}
    lon = []
    lat = []
    RM  = []
    
    for i in range(0,catalog[1].data.shape[0]):
        lon.append(catalog[1].data[i][2])
        lat.append(catalog[1].data[i][3])
        RM.append( catalog[1].data[i][5])
    
    st_cat['lon'] = np.array(lon)
    st_cat['lat'] = np.array(lat)
    st_cat['RM'] = np.array(RM)
    
    return st_cat

In [ ]:
def make_4panel_linfit(l_list, b_list, outname='test.pdf'):

    fs = 15
    s = 8
    lbd2_ext = np.linspace(0.0435,0.0456,100)

    fig, axs = plt.subplots(2,2,figsize=(12,12))
    plt.subplots_adjust(left=0.1, bottom=0.05, right=0.99, top=0.95, wspace=0.1, hspace=0.1)

    panels = ['(a)','(b)','(c)','(d)']
    
    for i in range(0,2):
        for j in range(0,2):
            
            k = 2*i + j
            print(k)

            l_idx0 = np.where(abs(l-l_list[k])<dlb/2.)[0][0]
            b_idx0 = np.where(abs(b-b_list[k])<dlb/2.)[0][0]

            print(l[l_idx0])
            print(b[b_idx0])

            PA_C_fix = []
            PA_G_fix = []
            PA_CG_fix = []
            PA_C_err = []
            PA_G_err = []
            PA_CG_err = []

            for kk in range(0,4):

                PA_C_fix.append(PA_C_list[kk][0].data[b_idx0,l_idx0])
                PA_G_fix.append(PA_G_list[kk][0].data[b_idx0,l_idx0])
                PA_CG_fix.append(PA_CG_list[kk][0].data[b_idx0,l_idx0])

                PA_C_err.append(PA_C_list[kk][1].data[b_idx0,l_idx0])
                PA_G_err.append(PA_G_list[kk][1].data[b_idx0,l_idx0])
                PA_CG_err.append(PA_CG_list[kk][1].data[b_idx0,l_idx0])

            PAint_pt_G = PAint_G[b_idx0,l_idx0]
            PAint_pt_C = PAint_C[b_idx0,l_idx0]
            PAint_pt_CG = PAint_CG[b_idx0,l_idx0]
            RM_pt_G = G_RM[b_idx0,l_idx0]
            RM_pt_C = C_RM[b_idx0,l_idx0]
            RM_pt_CG = CG_RM[b_idx0,l_idx0]
            err_pt_CG = stderr_CG[b_idx0,l_idx0]
            err_pt_G = stderr_G[b_idx0,l_idx0]
            err_pt_C = stderr_C[b_idx0,l_idx0]

            axs[i,j].errorbar(lbd2,np.array(PA_C_fix)*180/np.pi,yerr=np.array(PA_C_err)*180/np.pi,color='blue',fmt="o",
                              capsize=5,elinewidth=2,ms=s)
            axs[i,j].errorbar(lbd2,np.array(PA_G_fix)*180/np.pi,yerr=np.array(PA_G_err)*180/np.pi,color='orange',fmt="o",
                              capsize=5,elinewidth=2,ms=s)
            axs[i,j].errorbar(lbd2,np.array(PA_CG_fix)*180/np.pi,yerr=np.array(PA_CG_err)*180/np.pi,color='red',fmt="o",
                              capsize=5,elinewidth=2,ms=s)

            axs[i,j].plot(lbd2_ext,PAint_pt_C*180/np.pi+RM_pt_C*lbd2_ext*180/np.pi,color='blue',
                          label='RM$_{AS}$ = '+str(np.round(RM_pt_C,1))+'$\pm$'+str(np.round(err_pt_C,1))+' rad m$^{-2}$',
                         linewidth=2)
            
            axs[i,j].plot(lbd2_ext,PAint_pt_G*180/np.pi+RM_pt_G*lbd2_ext*180/np.pi,color='orange',
                          label='RM$_{SA}$ = '+str(np.round(RM_pt_G,1))+'$\pm$'+str(np.round(err_pt_G,1))+' rad m$^{-2}$',
                         linewidth=2)
            
            axs[i,j].plot(lbd2_ext,PAint_pt_CG*180/np.pi+RM_pt_CG*lbd2_ext*180/np.pi,color='red',
                          label='RM$_{SA+AS}$ = '+str(np.round(RM_pt_CG,1))+'$\pm$'+str(np.round(err_pt_CG,1))+' rad m$^{-2}$',
                         linewidth=2)

            if k == 0:
                y1 = -45
                y2 = 135
            else:
                y1 = -180
                y2 = 0
            axs[i,j].text(0.04355,y2-10, panels[k], fontsize=fs+2, bbox={'facecolor':'white', 'alpha':1, 'edgecolor':'k'}, 
                          ha='center', va='center')
            
            axs[i,j].set_xlim(0.0434,0.0456)
            axs[i,j].set_xticks([0.0435,0.044,0.0445,0.045,0.0455])
            axs[i,j].set_yticks([-180,-135,-90,-45,0,45,90,135,180])
            
            if i == 1:
                axs[i,j].set_xticklabels(['0.0435','0.044','0.0445','0.045','0.0455'])
                axs[i,j].set_xlabel(r'$\lambda^2$ (m$^2$)',fontsize=fs)
            else:
                axs[i,j].set_xticklabels(['','','','',''])
            if j == 0:
                axs[i,j].set_yticklabels(['-180','-135','-90','-45','0','45','90','135','180'])
                axs[i,j].set_ylabel('PA (degrees)',fontsize=fs)
            else:
                axs[i,j].set_yticklabels(['','','','','','','','',''])

            axs[i,j].set_ylim(y1,y2)
            axs[i,j].grid()
            axs[i,j].tick_params(axis='both', labelsize=fs, left=True, right=True, which='both', width=2, length=6)
            axs[i,j].legend(fontsize=fs)
            axs[i,j].set_title('$\ell$ = '+str(np.round(l[l_idx0],3))+r'$^{\circ}$, $b$ = '+str(np.round(b[b_idx0],3))+r'$^{\circ}$',
                              fontsize=fs+2)
            for spine in axs[i,j].spines.values():
                spine.set_visible(True)
                spine.set_linewidth(2)

    plt.savefig(outname)

    return

In [ ]:
def make_long_bins(datasets, lon_sets, lmin=54, lmax=190, dl=1.5):

    lbins = np.arange(lmin,lmax+dl,dl)

    binned_all = []
    stddev_all  = []

    for j in range(0,len(datasets)):
        #print(datasets[j].shape)
        
        binned = np.empty_like(lbins)
        stddev = np.empty_like(lbins)
    
        for i in range(0,len(lbins)):
    
            idx = np.where((lon_sets[j] >= lbins[i]-dl/2) & (lon_sets[j] < lbins[i]+dl/2))[0]
            #print(lbins[i],len(idx))

            if datasets[j].ndim == 2:
                binned[i] = np.nanmean(datasets[j][:,idx])
                stddev[i] = np.nanstd(datasets[j][:,idx])
            else:
                binned[i] = np.nanmedian(datasets[j][idx])
                stddev[i] = np.nanstd(datasets[j][idx])
        #print('')

        binned_all.append(binned)
        stddev_all.append(stddev)

    return lbins, binned_all, stddev_all

In [ ]:
def RM_versus_longitude(lbins, binned_all, stddev_all, outname = 'test.pdf'):

    fig,ax = plt.subplots(3,1,figsize = (20,14))
    plt.subplots_adjust(hspace=0.1)
    
    fs = 24
    lw = 1.5

    panels = ['(a)','(b)','(c)']
    ylims = [600,400,100]
    
    ###############################################################################
    ax[0].plot(lbins,binned_all[4],label='CGPS EG RM', color='C2',linewidth=lw)
    ax[0].fill_between(lbins, binned_all[4]-stddev_all[4], binned_all[4]+stddev_all[4], color='C2', alpha=0.4)
    ax[0].scatter(lbins,binned_all[4], color='C2',zorder=100)
    
    ax[0].plot(lbins,binned_all[0],label='CGPS XE RM', color='C0',linestyle='dashed',linewidth=lw)
    ax[0].fill_between(lbins, binned_all[0]-stddev_all[0], binned_all[0]+stddev_all[0], color='C0', alpha=0.4)
    ax[0].scatter(lbins,binned_all[0], color='C0',zorder=100)
    
    ax[0].axhline(y=0,color='k',zorder=10)
    
    ###############################################################################
    ax[1].plot(lbins,binned_all[1],label='GMIMS XE RM', color='C1',linewidth=lw)
    ax[1].fill_between(lbins, binned_all[1]-stddev_all[1], binned_all[1]+stddev_all[1], color='C1', alpha=0.4)
    ax[1].scatter(lbins,binned_all[1], color='C1',zorder=100)
    
    ax[1].plot(lbins,binned_all[2],label='CGPS+GMIMS XE RM', color='C3',linestyle='dashed',linewidth=lw)
    ax[1].fill_between(lbins, binned_all[2]-stddev_all[2], binned_all[2]+stddev_all[2], color='C3', alpha=0.4)
    ax[1].scatter(lbins,binned_all[2], color='C3',zorder=100)
    
    ax[1].axhline(y=0,color='k',zorder=10)
    
    ###############################################################################
    ax[2].plot(lbins,binned_all[3],label='GMIMS XE FD', color='C4',linewidth=lw)
    ax[2].fill_between(lbins, binned_all[3]-stddev_all[3], binned_all[3]+stddev_all[3], color='C4', alpha=0.4)
    ax[2].scatter(lbins,binned_all[3], color='C4',zorder=100)
    
    ax[2].axhline(y=0,color='k',zorder=10)
    
    ###############################################################################
    for i in range(0,3):
        ax[i].set_xlim(192,50)
        ax[i].grid()
        ax[i].legend(loc='lower left',fontsize=fs)
        ax[i].set_ylabel(r'RM (rad m$^{-2}$)',fontsize=fs)
        ax[i].set_xticklabels([])
        ax[i].set_xticks([190,180,170,160,150,140,130,120,110,100,90,80,70,60,50])
        ax[i].set_ylim(-ylims[i],ylims[i])
        ax[i].tick_params(axis='both', labelsize=fs, left=True, right=False, which='both', width=2, length=6)
        for spine in ax[i].spines.values():
            spine.set_visible(True)
            spine.set_linewidth(2)
        ax[i].text(185,0.8*ylims[i], panels[i], fontsize=fs+2, bbox={'facecolor':'white', 'alpha':1, 'edgecolor':'k'}, 
                          ha='center', va='center')
    ax[2].set_xlabel('Galactic Longitude (degrees)',fontsize=fs)
    ax[2].set_ylabel(r'FD (rad m$^{-2}$)',fontsize=fs)
    ax[2].set_xticklabels([190,180,170,160,150,140,130,120,110,100,90,80,70,60,50],fontsize=fs);
    
    plt.tight_layout()
    plt.savefig(outname)

    return

## Read in RM files and set to NaN where RM=0 (blank around map edges):

In [ ]:
hdu_CG_RM = fits.open(dir_in+'RM_CG_conv4_regrd.fits')

CG_RM     = hdu_CG_RM[0].data
PAint_CG  = hdu_CG_RM[1].data
rvalue_CG = hdu_CG_RM[2].data
stderr_CG = hdu_CG_RM[4].data

CG_RM[hdu_CG_RM[0].data==0] = np.nan
PAint_CG[hdu_CG_RM[0].data==0] = np.nan
rvalue_CG[hdu_CG_RM[0].data==0] = np.nan
stderr_CG[hdu_CG_RM[0].data==0] = np.nan

print(CG_RM.shape)

##########################################################

hdu_G_RM = fits.open(dir_in+'RM_G_regrd.fits')

G_RM     = hdu_G_RM[0].data
PAint_G  = hdu_G_RM[1].data
rvalue_G = hdu_G_RM[2].data
stderr_G = hdu_G_RM[4].data

G_RM[hdu_G_RM[0].data==0] = np.nan
PAint_G[hdu_G_RM[0].data==0] = np.nan
rvalue_G[hdu_G_RM[0].data==0] = np.nan
stderr_G[hdu_G_RM[0].data==0] = np.nan

print(G_RM.shape)

##########################################################

hdu_C_RM = fits.open(dir_in+'RM_C_conv4_regrd.fits')

C_RM     = hdu_C_RM[0].data
PAint_C  = hdu_C_RM[1].data
rvalue_C = hdu_C_RM[2].data
stderr_C = hdu_C_RM[4].data

C_RM[hdu_C_RM[0].data==0] = np.nan
PAint_C[hdu_C_RM[0].data==0] = np.nan
rvalue_C[hdu_C_RM[0].data==0] = np.nan
stderr_C[hdu_C_RM[0].data==0] = np.nan

print(C_RM.shape)


## Read in PA files:

In [ ]:
PA_CG_list = []
PA_G_list = []
PA_C_list = []

band = ['A','B','C','D']
bandlc = ['a','b','c','d']
for i in range(0,4):
    
    print('band '+band[i])
    
    # CGPS + GMIMS (CG)
    directory = '/srv/data/cgps-gmims/conv_regrid/'
    hdu_CG = fits.open(directory+'PA_'+band[i]+'_CG_conv4_regrd.fits')
    PA_CG_list.append(hdu_CG)
    
    # Grab a header (all the same for now)
    hdr = hdu_CG[0].header
    
    # CGPS only (C)
    directory = '/srv/data/cgps-gmims/conv_regrid/'
    hdu_C = fits.open(directory+'PA_'+band[i]+'_C_conv4_regrd.fits')
    PA_C_list.append(hdu_C)
    
    # GMIMS only (G)
    directory = '/srv/data/cgps-gmims/conv_regrid/'
    hdu_G = fits.open(directory+'PA_'+band[i]+'_G_regrd.fits')
    PA_G_list.append(hdu_G)
    
gc.collect()


## Read in PI files and set to NaN where RM=0 (blank around map edges):

In [ ]:
hdu_CG_PI = fits.open(dir_in+'PI_CG_conv4_regrd_PI_of_mean.fits')
CG_PI = hdu_CG_PI[0].data
CG_PI[hdu_CG_RM[0].data==0] = np.nan
print(CG_PI.shape)

hdu_G_PI = fits.open(dir_in+'PI_G_regrd_PI_of_mean.fits')
G_PI = hdu_G_PI[0].data
G_PI[hdu_G_RM[0].data==0]
print(G_PI.shape)

hdu_C_PI = fits.open(dir_in+'PI_C_conv4_regrd_PI_of_mean.fits')
C_PI = hdu_C_PI[0].data
C_PI[hdu_C_RM[0].data==0]
print(C_PI.shape)

hdr = hdu_C_PI[0].header

## Get point source RM list and GMIMS FD

In [ ]:
st_cat = read_st_catalogs('/srv/data/cgps/', 'CGPS_RMTable.fits')

FD_G = fits.open('/srv/data/cgps-gmims/gmims_FD/phi_peak_regrd.fits')[0].data

print(FD_G.shape)

## Set low PI and high-error RM to zero

In [ ]:
C_RM[C_PI < P_thr] = np.nan
C_RM[stderr_C > dRM_thr] = np.nan

CG_RM[CG_PI < P_thr] = np.nan
CG_RM[stderr_CG > dRM_thr] = np.nan

G_RM[G_PI < P_thr] = np.nan
G_RM[stderr_G > dRM_thr] = np.nan

FD_G[G_PI < P_thr] = np.nan


## Get lambda^2, longitudes, latitudes

In [ ]:
freq = np.array([1406.9,1413.8,1427.4,1434.3])
lbd2 = ((3e8)/(freq*1e6))**2

l = WCS(hdr).all_pix2world(range(PA_CG_list[0][0].data.shape[1]) ,0, 0)[0]
b = WCS(hdr).all_pix2world(0, range(PA_CG_list[0][0].data.shape[0]), 0)[1]

dlb = l[0]-l[1]

print(l)
print(b)


# The plots

## Example of linear fits (Figure 4)

In [ ]:
make_fig4 = False

if make_fig4:

    l_list = [125.79, 188.84, 93.56, 134.61]
    b_list = [0.97,   2.54,  -0.32,  -1.51]
    
    make_4panel_linfit(l_list, b_list, '../plots/sample_linear_fits_errorbars.pdf')
    

## RM versus longitude (Figure 13)

In [ ]:
make_fig13 = True

if make_fig13:

    datasets = [C_RM, G_RM, CG_RM, FD_G, st_cat['RM'][st_cat['lat']<5.5]]
    lon_sets = [l, l, l, l, st_cat['lon'][st_cat['lat']<5.5]]
    
    lbins,binned_all,stddev_all = make_long_bins(datasets, lon_sets, lmin=52, lmax=190, dl=1.5)

    RM_versus_longitude(lbins, binned_all, stddev_all, outname = '../plots/RM_versus_long_new.pdf')
